# Steam Market Intelligence Report

**Module:** M504 AI and Applications  
**Author:** Ashish Negi  
**Client:** A new indie game studio planning to release its first game on Steam.  
**Project goal:** Use historical Steam data to recommend pricing, platform support, review strategy, engagement targets, launch expectations, and realistic KPIs.

## Executive Summary

This report analyses historical Steam marketplace data to support strategic decisions for a new indie game studio preparing its first game release on Steam.

**Key findings:**
- Affordable pricing (below $15) correlates with the broadest player reach across the Steam catalogue.
- Games with consistently positive user reviews are associated with higher estimated ownership.
- Windows remains the dominant platform, representing over 79% of all Steam titles.
- Steam market competition has increased approximately 25-fold since 2013, making discoverability a primary challenge.
- The majority of games fall within the 0–20,000 estimated owners bracket, underscoring the importance of realistic launch targets.

**Strategic recommendation:** The studio should prioritise a low-risk, accessibility-focused launch strategy — affordable pricing, Windows-first development, early review generation, and a pre-launch community. Ownership expectations should be benchmarked against realistic percentile targets rather than viral outliers.

## Dataset Source and Business Context

- Dataset: Steam games market dataset from Kaggle.
- Source URL: https://www.kaggle.com/datasets/fronkongames/steam-games-dataset
- The client is a first-time indie studio preparing a Steam launch and needs evidence-based guidance on pricing, platforms, reviews, engagement, and launch positioning.

## Methodology

This analysis follows a standard data science pipeline applied to a real-world commercial dataset:

1. **Data acquisition** — sourcing and loading the Steam games dataset from Kaggle.
2. **Data quality assessment** — identifying missing values, duplicates, and structural inconsistencies.
3. **Data preprocessing** — converting raw fields into analytically useful features (ownership midpoints, price tiers, platform categories, review metrics).
4. **Exploratory data analysis** — examining distributions, trends, and patterns across key commercial variables.
5. **Business question investigation** — eight targeted questions addressing the studio's specific strategic decisions.
6. **Advanced analysis** — correlation analysis to identify which factors most strongly relate to ownership success.
7. **Recommendation generation** — evidence-based guidance derived from each analytical finding.

**Tools used:** Python 3, Pandas, NumPy, and Plotly Express for interactive visualisation. All analysis was conducted in a reproducible Jupyter Notebook environment.

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "iframe"
pd.options.display.max_columns = None

## Data Loading and Initial Validation

Load the Steam dataset with corrected headers, verify the number of records, and inspect the first rows.

In [10]:
cols = [
    "AppID", "Name", "Release date", "Estimated owners", "Peak CCU",
    "Required age", "Price", "Discount", "DLC count", "About the game",
    "Supported languages", "Full audio languages", "Reviews", "Header image",
    "Website", "Support url", "Support email", "Windows", "Mac", "Linux",
    "Metacritic score", "Metacritic url", "User score", "Positive", "Negative",
    "Score rank", "Achievements", "Recommendations", "Notes",
    "Average playtime forever", "Average playtime two weeks",
    "Median playtime forever", "Median playtime two weeks", "Developers",
    "Publishers", "Categories", "Genres", "Tags", "Screenshots", "Movies"
]

df = pd.read_csv('games.csv', header=None, names=cols, skiprows=1, engine='python')
print('Dataset rows:', len(df))
print('Dataset columns:', len(df.columns))
print(df.columns.tolist())
df.head(3)

Dataset rows: 122611
Dataset columns: 40
['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'Discount', 'DLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,NaN,0,21,3,NaN,0,0,NaN,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN


## Dataset Description

The dataset includes Steam game metadata and commercial metrics such as price, estimated owners, review counts, platform availability, playtime, and launch dates. This information supports decisions for pricing, platform support, review strategy, engagement expectations, and launch targets.

In [11]:
description = {
    'Games': len(df),
    'Columns': len(df.columns),
    'Unique prices': df['Price'].nunique(),
    'Estimated owners categories': df['Estimated owners'].nunique(),
    'Release years': df['Release date'].nunique(),
}
print(description)
print('Sample unique estimated owner categories:', df['Estimated owners'].unique()[:12])

{'Games': 122611, 'Columns': 40, 'Unique prices': 941, 'Estimated owners categories': 14, 'Release years': 5081}
Sample unique estimated owner categories: ['0 - 0' '0 - 20000' '100000 - 200000' '500000 - 1000000' '20000 - 50000'
 '200000 - 500000' '50000 - 100000' '2000000 - 5000000'
 '10000000 - 20000000' '1000000 - 2000000' '20000000 - 50000000'
 '5000000 - 10000000']


## Data Quality Assessment

Identify missing values, duplicate records, inconsistent data formats, and quality issues that must be addressed before analysis.

In [12]:
missing = df.isnull().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()
print('Duplicate rows:', duplicates)
print('Top missing columns:', missing[missing > 0].head(12))
print('Estimated owners type:', df['Estimated owners'].dtype)
print('Price type:', df['Price'].dtype)

Duplicate rows: 0
Top missing columns: Movies            122611
Score rank        122571
Metacritic url    118355
Reviews           110541
Notes             100153
Website            72935
Support url        68469
Tags               39265
Support email      22263
Categories          8953
Publishers          8909
About the game      8449
dtype: int64
Estimated owners type: object
Price type: float64


### Data Quality Findings

- **No duplicate records** were identified across the 122,611 game entries, indicating a clean primary key structure.
- **Ownership values are estimated ranges** (e.g., "20,000 – 50,000") rather than exact sales figures. This introduces imprecision and all ownership-based analysis should be interpreted as approximate.
- **Several columns have substantial missing data:** `Movies`, `Score rank`, `Metacritic url`, and `Reviews` are largely absent and cannot be used for analysis. These were excluded from the EDA pipeline.
- **Metadata columns** (`Website`, `Support url`, `Tags`, `Categories`) have moderate missingness (30–65%) — treated as optional fields and used only where populated.
- **Ownership encoded as string ranges** requires preprocessing to convert to numeric midpoints before any quantitative comparison can be made.
- **No severe data corruption** was detected in the key commercial columns (Price, Positive, Negative, Peak CCU, Platform flags).

### Potential Biases

- **Ownership estimation bias:** Because ownership is reported as a wide range (e.g., "0 – 20,000"), a game with 1 owner and one with 19,999 owners are treated identically in aggregation. Midpoint conversion partially addresses this but does not eliminate the imprecision.
- **Survivor bias in engagement metrics:** Only games with recorded playtime (approximately 21% of the dataset) contribute to engagement analysis. Games that flopped immediately after launch may underreport or not report playtime at all, making the engagement picture more optimistic than the full market reality.
- **Recency bias in new releases:** Games released in 2024–2025 may not yet have accumulated representative review counts or engagement data, as player behaviour and word-of-mouth take time to accumulate. Their metrics may appear lower than they will be once the games mature.

## Data Preprocessing

Convert ownership ranges to numeric midpoints, extract release year, compute review metrics, classify price tiers, and define platform support categories.

In [21]:
def parse_owner_range(value):
    if isinstance(value, str) and ' - ' in value:
        low, high = value.split(' - ')
        try:
            return (int(low.replace(',', '')) + int(high.replace(',', ''))) / 2
        except ValueError:
            return np.nan
    return np.nan

df['Estimated owners midpoint'] = df['Estimated owners'].astype(str).apply(parse_owner_range)
df['Release Year'] = pd.to_datetime(df['Release date'], errors='coerce').dt.year
df['Review count'] = df['Positive'] + df['Negative']
df['Positive ratio'] = np.where(df['Review count'] > 0, df['Positive'] / df['Review count'], np.nan)

def price_category(price):
    if price == 0:
        return 'Free'
    if price < 10:
        return 'Low'
    if price < 30:
        return 'Mid'
    return 'High'

df['Price Category'] = df['Price'].apply(price_category)

def platform_support(row):
    platforms = []
    if row['Windows']:
        platforms.append('Windows')
    if row['Mac']:
        platforms.append('Mac')
    if row['Linux']:
        platforms.append('Linux')
    if len(platforms) == 0:
        return 'None'
    if len(platforms) == 1:
        return platforms[0] + ' Only'
    if len(platforms) == 2:
        return '+'.join(platforms)
    return 'All platforms'

df['Platform Support'] = df.apply(platform_support, axis=1)

print(df[['Estimated owners', 'Estimated owners midpoint', 'Price', 'Price Category', 'Platform Support', 'Positive ratio', 'Release Year']])

       Estimated owners  Estimated owners midpoint  Price Price Category  \
0                 0 - 0                        0.0   0.00           Free   
1             0 - 20000                    10000.0   5.24            Low   
2             0 - 20000                    10000.0   4.99            Low   
3             0 - 20000                    10000.0   8.99            Low   
4             0 - 20000                    10000.0   4.99            Low   
...                 ...                        ...    ...            ...   
122606            0 - 0                        0.0   0.00           Free   
122607            0 - 0                        0.0   0.00           Free   
122608            0 - 0                        0.0   7.99            Low   
122609            0 - 0                        0.0   2.09            Low   
122610            0 - 0                        0.0   0.00           Free   

       Platform Support  Positive ratio  Release Year  
0          Windows Only        

## Q1. What pricing strategy should the studio adopt?

**Business value:** The launch price directly impacts discovery, sales, and player adoption.

**Analysis plan:** Compare price categories by ownership and review feedback to find the pricing zones that offer strong reach without sacrificing quality.

In [29]:
price_summary = df.groupby('Price Category', observed=True).agg(
    games=('AppID', 'size'),
    median_owners=('Estimated owners midpoint', 'median'),
    mean_positive_ratio=('Positive ratio', 'mean'),
    median_review_count=('Review count', 'median')
).reset_index()
price_summary

,Price Category,games,median_owners,mean_positive_ratio,median_review_count
0,Free,26206,0.0,0.754700,0.0
1,High,1094,10000.0,0.626434,3.0
2,Low,84605,10000.0,0.760198,10.0
3,Mid,10706,10000.0,0.757844,20.0


In [30]:
# Prepare labels safely
price_summary['median_owners_safe'] = price_summary['median_owners'].fillna(0)
price_summary['label'] = price_summary['median_owners_safe'].apply(lambda x: '' if x==0 else f'{int(x):,}')
# Decide axis scale
use_log = (price_summary['median_owners_safe'] > 0).all()
fig = px.bar(
    price_summary,
    x='Price Category',
    y='median_owners_safe',
    text='label',
    title='Median Estimated Owners by Price Category',
    labels={'median_owners_safe': 'Median Estimated Owners', 'Price Category': 'Price Category'}
)
if use_log:
    fig.update_layout(yaxis_type='log')
fig.update_traces(textposition='outside')
fig.show()


**Note:** The chart uses a logarithmic y-axis only when all median values are positive. Labels hide zero/NaN medians to avoid clutter and log errors.

### Interpretation

The chart shows that Free and Low-priced games dominate Steam by volume, and the median estimated ownership for Low-priced titles matches or exceeds that of higher price brackets. Mid-tier pricing ($10–$30) generates slightly better median review engagement, suggesting that paying players may be more personally invested in a title they have purchased at a higher price point. **This matters because price is the first signal a potential player sees on the Steam store page, and setting it too high can prevent the game from being clicked on at all — particularly for an unknown studio with no existing audience.**

### Business Insight

Price is a significant barrier to discovery on Steam. The majority of the catalogue is priced below $10, meaning that premium pricing immediately places an unknown studio at a visibility disadvantage. Steam's algorithm also factors in sales volume, so a lower price that generates more transactions can compound into better organic placement.

### Recommendation

Launch at an affordable price point, ideally $5–$15. Consider a temporary launch discount of 15–20% to maximise early adoption and build initial review momentum. Avoid premium pricing unless the game has exceptional production values and a pre-built audience from prior marketing activity.

## Q2. Which platforms should the game support at launch?

**Business value:** Development resources are limited, so platform support must maximize audience while minimizing complexity.

**Analysis plan:** Compare the count and ownership performance of Windows-only, dual-platform, and all-platform releases.

In [32]:
platform_summary = df.groupby('Platform Support', observed=True).agg(
    games=('AppID', 'size'),
    median_owners=('Estimated owners midpoint', 'median'),
    mean_positive_ratio=('Positive ratio', 'mean')
).reset_index().sort_values(by='games', ascending=False)
platform_summary

,Platform Support,games,median_owners,mean_positive_ratio
4,Windows Only,96918,10000.0,0.745469
0,All platforms,11303,10000.0,0.797251
6,Windows+Mac,9957,10000.0,0.799105
5,Windows+Linux,4389,10000.0,0.811333
2,Mac Only,30,10000.0,0.447868
1,Linux Only,12,10000.0,0.667033
3,Mac+Linux,2,22500.0,0.848921


In [33]:
fig = px.bar(
    platform_summary,
    x='games',
    y='Platform Support',
    orientation='h',
    text='games',
    title='Steam Games by Platform Support',
    labels={'games': 'Number of Games', 'Platform Support': 'Platform Support'}
)
fig.update_layout(yaxis={'categoryorder':'total descending'})
fig.show()

### Interpretation

Windows-only titles represent approximately 79% of all Steam games. Multi-platform titles (Windows+Mac, All platforms) show slightly higher average positive review ratios, suggesting a modest correlation between broader platform support and community reception — though this may reflect higher-budget productions rather than the platform choice itself. **This matters because supporting additional platforms at launch increases development cost and testing overhead while only marginally expanding the addressable market for a first-time release.**

### Business Insight

The Steam player base is overwhelmingly Windows-centric. Supporting additional platforms requires additional development effort, testing pipelines, and certification processes that may not be justified by the incremental audience gain for a first-time studio operating on constrained resources.

### Recommendation

Launch on Windows first and direct all quality assurance resources there. Add Mac support as a post-launch milestone if the game engine makes cross-compilation straightforward. Linux support can be evaluated later based on community demand and Steam Deck compatibility interest.

## Q3. How important are user reviews for commercial success?

**Business value:** If review sentiment strongly correlates with ownership, the studio should make review generation a launch priority.

**Analysis plan:** Compare ownership buckets with positive review ratios and review volume.

In [34]:
review_df = df[df['Review count'] > 0].copy()
review_df['owner_band'] = pd.qcut(review_df['Estimated owners midpoint'].rank(method='first'), 5, labels=['Lowest', 'Lower', 'Middle', 'Higher', 'Highest'])
review_by_band = review_df.groupby('owner_band', observed=True).agg(
    median_positive_ratio=('Positive ratio', 'median'),
    median_review_count=('Review count', 'median'),
    median_owners=('Estimated owners midpoint', 'median')
).reset_index()
review_by_band

,owner_band,median_positive_ratio,median_review_count,median_owners
0,Lowest,0.833333,14.0,10000.0
1,Lower,0.823529,14.0,10000.0
2,Middle,0.820000,14.0,10000.0
3,Higher,0.802975,29.0,35000.0
4,Highest,0.818833,642.0,150000.0


In [35]:
fig = px.line(
    review_by_band,
    x='median_owners',
    y='median_positive_ratio',
    markers=True,
    title='Positive Review Ratio vs. Estimated Ownership',
    labels={'median_owners': 'Median Estimated Owners', 'median_positive_ratio': 'Median Positive Review Ratio'}
)
fig.update_xaxes(type='log')
fig.show()

### Interpretation

Games in higher ownership bands accumulate significantly more reviews, but the positive review ratio remains relatively stable across all bands (approximately 80–83%). This suggests it is not simply having many reviews that drives ownership success — it is review quality and consistency maintained throughout the ownership spectrum. Games that sustain a high positive ratio as review volume grows tend to benefit most from Steam's recommendation system. **This matters because Steam's discovery algorithm actively promotes games with higher review counts and strong positive ratios, meaning early review generation has a compounding effect on organic visibility.**

### Business Insight

Reviews act as social proof and a direct input to Steam's recommendation algorithm. Even games with modest ownership benefit from maintaining a strong positive ratio. Early negative reviews create a lasting impression on the store page and are difficult to overcome without a significant subsequent volume of positive feedback.

### Recommendation

Prioritise review generation from launch day. Reach out to early access players, content creators, and streamers for pre-launch copies. Respond professionally to all negative feedback. Aim to accumulate a minimum of 10–15 reviews before the end of the launch weekend in order to establish an initial sentiment signal on the store page.

## Q4. What characteristics are shared by highly successful games?

**Business value:** Success factors can shape the studio’s product and launch strategy.

**Analysis plan:** Analyze the top 10% of games by estimated owners for price, engagement, review quality, and platform mix.

In [36]:
top_threshold = df['Estimated owners midpoint'].quantile(0.90)
top_games = df[df['Estimated owners midpoint'] >= top_threshold]

In [37]:
success_summary = pd.DataFrame({
    'Metric': [
        'Median Price',
        'Median Peak CCU',
        'Median Average Playtime',
        'Median Positive Ratio',
        'Windows Share',
        'Multi-platform Share'
    ],
    'Top 10%': [
        top_games['Price'].median(),
        top_games['Peak CCU'].median(),
        top_games['Average playtime forever'].median(),
        top_games['Positive ratio'].median(),
        (top_games['Platform Support'].str.contains('Windows')).mean(),
        top_games['Platform Support'].isin(['Windows+Mac', 'Windows+Linux', 'All platforms']).mean()
    ],
    'All games': [
        df['Price'].median(),
        df['Peak CCU'].median(),
        df['Average playtime forever'].median(),
        df['Positive ratio'].median(),
        (df['Platform Support'].str.contains('Windows')).mean(),
        df['Platform Support'].isin(['Windows+Mac', 'Windows+Linux', 'All platforms']).mean()
    ]
})
success_summary

,Metric,Top 10%,All games
0,Median Price,2.990000,2.240000
1,Median Peak CCU,2.000000,0.000000
2,Median Average Playtime,236.000000,0.000000
3,Median Positive Ratio,0.821979,0.818182
4,Windows Share,0.830769,0.907455
5,Multi-platform Share,0.324418,0.209190


In [38]:
print('Descriptive statistics for top 10% games:')
top_games[['Price', 'Peak CCU', 'Review count', 'Positive ratio', 'Average playtime forever']].describe().round(2)

Descriptive statistics for top 10% games:


,Price,Peak CCU,Review count,Positive ratio,Average playtime forever
count,14170.00,14170.00,14170.00,14152.00,14170.00
mean,5.24,444.24,9980.11,0.78,1086.09
std,7.14,10836.07,95288.19,0.17,29123.92
min,0.00,0.00,0.00,0.00,0.00
25%,0.87,0.00,236.00,0.70,52.00
50%,2.99,2.00,910.50,0.82,236.00
75%,6.99,21.00,3254.00,0.91,569.75
max,199.99,1013936.00,8815087.00,1.00,3429544.00


**Which metric separates successful games the most?**

The descriptive statistics above reveal that **Peak CCU** and **Review count** show the greatest absolute spread between the top 10% and the full catalogue. Average playtime also differentiates successful games, with the top 10% recording a median of approximately 236 hours versus zero for the full dataset (most games record no playtime data at all). Price, by contrast, is nearly indistinguishable between groups — confirming that price point alone is not a differentiating factor for commercial success. The strongest signal of a high-performing game is sustained concurrent engagement over time.

In [39]:
fig = px.bar(
    success_summary.melt(id_vars=['Metric'], var_name='Group', value_name='Value'),
    x='Metric',
    y='Value',
    color='Group',
    barmode='group',
    title='Top 10% Games vs Entire Catalog'
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

### Interpretation

Top-performing games differ from the median game primarily in peak concurrency, average playtime, and multi-platform availability. Notably, positive review ratios are similar across the top 10% and the full catalogue (~82%), indicating that review quality alone does not separate successful games from average ones. Engagement depth — how long and how often players return — is a more differentiating factor. **This matters because it means the studio's design decisions — content depth, replayability, difficulty curve — are among the most controllable levers available for increasing the probability of commercial success.**

### Business Insight

Sustained player engagement, reflected in playtime and concurrent users, distinguishes commercially successful games. Games that keep players returning generate more organic reviews, more word-of-mouth referrals, and better algorithmic placement on Steam Discovery. A short but excellent experience still risks being forgotten quickly.

### Recommendation

Design for retention from the earliest stage of development. Include content that rewards replay (procedural elements, multiple difficulty modes, branching paths, achievements). Track playtime and concurrent user metrics closely during the first 30 days as leading indicators of long-term commercial performance.

## Q5. How competitive is the Steam marketplace?

**Business value:** Market competitiveness affects discoverability, marketing investment, and timing.

**Analysis plan:** Analyze annual release growth and identify how crowded the market has become.

In [40]:
games_per_year = df[df['Release Year'].notna()].groupby('Release Year').size().reset_index(name='Games').sort_values('Release Year')
games_per_year.tail(12)

,Release Year,Games
18,2015,2515
19,2016,4141
20,2017,5920
21,2018,7461
22,2019,7242
23,2020,8804
24,2021,11067
25,2022,12284
26,2023,14596
27,2024,20031


In [41]:
fig = px.line(
    games_per_year,
    x='Release Year',
    y='Games',
    markers=True,
    title='Number of Steam Game Releases per Year'
)
fig.update_layout(xaxis_title='Release Year', yaxis_title='Number of Games Released')
fig.show()

### Interpretation

Steam releases grew from fewer than 1,000 per year in 2013 to nearly 25,000 by 2025 — a 25-fold increase in competitive volume over just over a decade. This trend shows no sign of reversal. Every year the studio delays its launch, it enters an even more crowded market. **This matters because a game launched today without an established audience or a marketing plan competes against hundreds of other new releases in the same week, most of which are invisible to the average player within days.**

### Business Insight

Being a technically good game is no longer sufficient for commercial success on Steam. With thousands of simultaneous releases competing for player attention each month, marketing, community building, and Steam algorithm optimisation (tags, store page quality, wishlist accumulation) have become as important as the game itself.

### Recommendation

Build a community before launch. Set up a Steam store page and wishlist campaign at least three months prior to release. Maintain active social media and Discord presence. Plan for a dedicated launch-week marketing push. Target a launch window during periods of relatively lower competition — avoid major industry events and established release seasons dominated by AAA titles.

## Q6. What level of player engagement is typical?

**Business value:** Engagement expectations guide design, retention, and long-term roadmap planning.

**Analysis plan:** Review typical average playtime and peak concurrency values for released games.

In [42]:
engagement = df[df['Average playtime forever'] > 0]

In [43]:
playtime_stats = engagement['Average playtime forever'].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
peak_stats = engagement['Peak CCU'].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
print('Average playtime distribution:', playtime_stats)
print('Peak CCU distribution:', peak_stats)

Average playtime distribution: count    2.611900e+04
mean     9.765280e+02
std      2.428959e+04
min      1.000000e+00
25%      8.600000e+01
50%      2.170000e+02
75%      4.590000e+02
90%      1.204000e+03
max      3.429544e+06
Name: Average playtime forever, dtype: float64
Peak CCU distribution: count    2.611900e+04
mean     2.539582e+02
std      8.075425e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+00
75%      6.000000e+00
90%      5.500000e+01
max      1.013936e+06
Name: Peak CCU, dtype: float64


In [44]:
fig = px.histogram(
    engagement,
    x='Average playtime forever',
    nbins=40,
    title='Average Playtime Forever Distribution',
    labels={'Average playtime forever': 'Average Playtime (hours)'}
)
fig.update_yaxes(type='log')
fig.show()

### Interpretation

The playtime distribution is heavily right-skewed. The median average playtime for games with any recorded engagement is approximately 217 hours, but this figure is driven upward by a small number of extremely high-engagement outliers. For most games, typical players invest far less time. Peak concurrency for 75% of games with engagement data is six players or fewer simultaneously — a figure that would surprise many first-time developers. **This matters because resource allocation decisions — how much post-launch content to develop, whether to invest in multiplayer infrastructure — should be benchmarked against the typical player experience, not the exceptional outlier.**

### Business Insight

Most Steam games achieve modest engagement metrics. This is not a commercial failure — it is the statistical norm for the platform. Setting unrealistic engagement targets leads to poor resource prioritisation, unnecessary post-launch feature investment, and demoralisation within the team.

### Recommendation

Set a 90-day engagement KPI of 2–6 hours median average playtime for a typical indie title. Track median playtime rather than mean to avoid distortion from outliers. Monitor 30-day player retention rate as a leading indicator of long-term engagement health and use it to trigger targeted post-launch updates.

## Q7. What ownership level should a new studio realistically target?

**Business value:** A precise target helps the studio measure performance and manage expectations.

**Analysis plan:** Use ownership percentiles and common ownership bins to define realistic targets for a first-time indie launch.

In [45]:
ownership_percentiles = np.percentile(df['Estimated owners midpoint'].dropna(), [50, 75, 90, 95, 99])
print('Ownership percentiles (midpoint):', ownership_percentiles)

owner_range_counts = df['Estimated owners'].value_counts(normalize=True).mul(100).rename('Percent').reset_index().rename(columns={'index': 'Owner Range'}).sort_values('Percent', ascending=False)
owner_range_counts.head(8)

Ownership percentiles (midpoint): [  10000.   10000.   75000.  150000. 1500000.]


,Estimated owners,Percent
0,0 - 20000,61.498560
1,0 - 0,17.650129
2,20000 - 50000,9.294435
3,50000 - 100000,4.367471
4,100000 - 200000,2.817039
5,200000 - 500000,2.326871
6,500000 - 1000000,0.941188
7,1000000 - 2000000,0.594563


In [46]:
fig = px.bar(
    owner_range_counts.head(8),
    x='Percent',
    y='Estimated owners',
    orientation='h',
    text=owner_range_counts['Percent'].head(8).apply(lambda x: f'{x:.1f}%'),
    title='Distribution of Steam Games by Estimated Ownership Range',
    labels={'Percent': 'Percentage of Games (%)', 'Estimated owners': 'Ownership Range'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.update_traces(textposition='outside')
fig.show()

### Interpretation

Over 61% of Steam games fall within the 0–20,000 estimated owners bracket, and a further 18% report zero owners. Less than 5% of games ever exceed 100,000 owners. The 90th percentile ownership midpoint is approximately 75,000 — meaning nine out of ten Steam games never surpass that level of reach across their entire lifetime. **This matters because setting ownership targets based on viral outliers leads to poor financial planning and can cause a studio to prematurely declare a launch a failure when it is, in fact, performing at or above the market norm.**

### Business Insight

Ownership on Steam is highly concentrated among a small number of breakout titles. For an indie studio, contextualising success against realistic percentile benchmarks is critical to sustainable planning and honest investor or stakeholder communication. Comparing against viral outliers creates false expectations and can lead to premature decisions about the studio's viability.

### Recommendation

Define success in tiers based on the market data:
- **Tier 1 (0–20,000 owners):** Cost recovery — covers basic development and distribution costs.
- **Tier 2 (20,000–100,000 owners):** Sustainable studio — funds the next project.
- **Tier 3 (100,000+ owners):** Growth — enables team expansion and broader ambitions.

Use these tiers to set quarterly targets and communicate progress transparently. Plan the business model around Tier 1 outcomes and treat anything above as upside.

## Q8. What launch recommendations can be derived from historical Steam data?

Combine the findings into a focused launch strategy for the studio.

- Price the launch title in an affordable segment, ideally Free/Low/Mid rather than premium.
- Launch on Windows first, then consider Mac/Linux only if resources permit.
- Make positive review momentum a priority from day one.
- Expect modest average playtime and concurrency; design for replay and retention.
- Prepare a marketing and visibility plan because Steam is highly competitive.
- Use the 50th–75th percentile ownership range as a realistic KPI for an initial release.

## Q9. Which factors show the strongest relationship with estimated ownership?

**Business value:** Understanding which game attributes correlate most strongly with ownership helps the studio prioritise its pre-launch investments.

**Analysis plan:** Compute a correlation matrix across the key numeric commercial variables and visualise it as a heatmap to identify the strongest associations with estimated ownership.

In [47]:
correlation_cols = [
    'Price',
    'Peak CCU',
    'Review count',
    'Positive ratio',
    'Average playtime forever',
    'Estimated owners midpoint'
]

corr_df = df[correlation_cols].dropna()
corr_matrix = corr_df.corr()

print('Correlation matrix:')
print(corr_matrix.round(3))

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Matrix: Key Commercial Variables'
)
fig.update_layout(width=700, height=600)
fig.show()

Correlation matrix:
                           Price  Peak CCU  Review count  Positive ratio  \
Price                      1.000     0.006         0.013          -0.039   
Peak CCU                   0.006     1.000         0.873           0.005   
Review count               0.013     0.873         1.000           0.019   
Positive ratio            -0.039     0.005         0.019           1.000   
Average playtime forever   0.003     0.021         0.022          -0.000   
Estimated owners midpoint  0.014     0.647         0.760           0.020   

                           Average playtime forever  Estimated owners midpoint  
Price                                         0.003                      0.014  
Peak CCU                                      0.021                      0.647  
Review count                                  0.022                      0.760  
Positive ratio                               -0.000                      0.020  
Average playtime forever                  

### Interpretation

The correlation heatmap reveals that **Review count** and **Peak CCU** show the strongest positive associations with estimated ownership, while **Price** and **Positive ratio** show weaker direct correlations. This indicates that volume of engagement (how many people reviewed the game, how many played concurrently) is more tightly linked to ownership than price point or review sentiment alone. **This matters because it re-prioritises where the studio should invest its limited pre-launch and post-launch resources — community momentum and concurrent play events outweigh marginal price optimisation as a strategy for reaching more players.**

Note that **correlation does not imply causation**: a game is not more successful *because* it has more reviews — rather, successful games tend to attract more reviews as a consequence of their wider reach. However, the direction of these relationships remains strategically informative when deciding which activities to prioritise.

### Business Insight

Generating reviews and maximising concurrent player events (launch sales, seasonal discounts, content updates) are the highest-leverage activities for increasing ownership-associated signals. Price has a weaker direct correlation with ownership than community momentum metrics.

### Recommendation

Invest in community events, streamer partnerships, and time-limited discount periods that drive concurrent player spikes. These not only increase visibility but also generate the review volume that correlates most strongly with broader ownership outcomes.

## Critical Evaluation

### Strengths

- **Large dataset:** 122,611 game records provide statistically robust findings across all EDA dimensions. Results are unlikely to be materially affected by small-sample noise.
- **Multiple business metrics:** The dataset covers pricing, ownership, user reviews, playtime, concurrent users, and platform support — all directly relevant to the client's strategic decisions.
- **Real-world Steam market data:** Findings are grounded in actual marketplace behaviour rather than simulated, survey-based, or synthetic data.
- **Structured analytical framework:** Nine business questions (including advanced correlation analysis) ensure all critical decision areas are addressed systematically.
- **Reproducible pipeline:** All preprocessing, analysis, and visualisation steps are documented and reproducible in this notebook.

### Limitations

- **Ownership is estimated:** The dataset provides ownership ranges rather than exact sales figures, introducing imprecision into all ownership-based analysis. Midpoint conversions are approximations.
- **No marketing or promotion data:** It is not possible to determine how much studios invested in marketing, making it impossible to evaluate the relationship between promotional spend and commercial success.
- **No genre-level controls:** This analysis does not segment by genre. A horror game and a strategy game compete in different sub-markets and may follow distinct pricing, review, and engagement patterns.
- **Correlation does not imply causation:** Observed associations (e.g., review count and ownership) cannot be interpreted as causal without additional experimental or longitudinal data.
- **Historical bias:** The dataset reflects past market conditions. Genre trends, platform preferences, algorithm changes, and pricing norms may shift over time, limiting forward-looking applicability.
- **Self-reported platform flags:** Windows, Mac, and Linux support fields reflect developer-declared support and may not always reflect fully functional or tested implementations.

### Future Work

- **Genre-level analysis:** Segment the dataset by genre to provide more precise recommendations tailored to specific game types (e.g., RPG, puzzle, action).
- **Machine learning prediction:** Build a predictive model for ownership range using price, platform support, review count, playtime, and other features as inputs.
- **Regional market analysis:** Incorporate regional pricing data and regional sales breakdowns to advise on global pricing and localisation strategy.
- **Time-series analysis:** Investigate whether specific release windows (e.g., seasonal periods, post-sale periods) correlate with better ownership outcomes.
- **Causal inference:** Design a quasi-experimental study (e.g., difference-in-differences around Steam algorithm changes) to move beyond correlation toward causal claims.

## Conclusion

Historical Steam data indicates that an indie studio should focus on affordable pricing, Windows-first support, strong review sentiment, and realistic engagement targets. A clear launch KPI and marketing plan are vital due to the competitive nature of the platform.

## References

Fronkon Games (2024) *Steam Games Dataset*. Kaggle. Available at: https://www.kaggle.com/datasets/fronkongames/steam-games-dataset (Accessed: 18 June 2026).

McKinney, W. (2017) *Python for Data Analysis: Data Wrangling with Pandas, NumPy, and IPython*. 2nd edn. Sebastopol, CA: O'Reilly Media.

Plotly Technologies Inc. (2015) *Collaborative data science*. Montréal, QC: Plotly Technologies Inc. Available at: https://plot.ly (Accessed: 18 June 2026).

Valve Corporation (2003) *Steam*. Available at: https://store.steampowered.com (Accessed: 18 June 2026).

VanderPlas, J. (2016) *Python Data Science Handbook: Essential Tools for Working with Data*. Sebastopol, CA: O'Reilly Media.